# Hybrid LNN — Iterative Revisit Loop

This notebook runs the **change → train → analyse → repeat** cycle on the hybrid
LNN + wave + analogical council model.

### What it does
1. **Reads the last 5 episode logs** and deep-analyses them:
   - LNN voter accuracy & confidence trend
   - Wave voter degradation (vote-loss rising in 2nd half?)
   - Analogical confidence (dead?), competition-weight frozen, CYP-dominance
   - Board-weight stability, overfitting gap, precedent logbook activity
2. **Picks the highest-impact strategy** from the evidence (priority-ordered).
   Baseline selection and promotion use **validation metrics** by default,
   while test metrics are kept for reporting only.
3. **Trains 40–60 epochs** with early stopping.
4. **Compares** the result against the rolling baseline; promotes if better.
5. **Repeats** up to `ITERATIONS` times.  
   After 3 consecutive failures the loop switches to **alternate exploration paths**
   (live wave/analogical vote inputs, high-entropy board reset, memory unlock).

### Cells
| # | Purpose |
|---|---|
| 1 | Clone / pull repo & install |
| 2 | Mount Google Drive |
| 3 | Configure paths & revisit parameters |
| 4 | Run the revisit loop |
| 5 | Display results summary |

In [ ]:
# ── Cell 1: clone / pull repo and install ────────────────────────────────────
import os, subprocess, sys

REPO     = 'https://github.com/Nikku03/enzyme_Software.git'
REPO_DIR = '/content/enzyme_Software'

if not os.path.exists(REPO_DIR):
    print('Cloning repo...')
    subprocess.run(f'git clone {REPO} {REPO_DIR}', shell=True, check=True)
else:
    print('Repo exists — pulling latest...')
    subprocess.run(f'git -C {REPO_DIR} pull --ff-only', shell=True, check=True)

# Install the package (editable) and core deps
subprocess.run(f'pip -q install -e {REPO_DIR}', shell=True, check=True)
subprocess.run(f'pip -q install -r {REPO_DIR}/requirements.txt', shell=True)

# Clear stale repo bytecode so latest script/trainer fixes are picked up
subprocess.run(f'find {REPO_DIR}/src -name "*.pyc" -delete', shell=True)
subprocess.run(f'find {REPO_DIR}/src -name "__pycache__" -type d -exec rm -rf {{}} +', shell=True)

print('Repo ready:', REPO_DIR)

In [ ]:
# ── Cell 2: mount Google Drive ────────────────────────────────────────────────
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Pre-create expected directories so later cells don't fail
    for d in [
        '/content/drive/MyDrive/enzyme_hybrid_lnn/checkpoints/hybrid_full_xtb',
        '/content/drive/MyDrive/enzyme_hybrid_lnn/artifacts/hybrid_full_xtb',
        '/content/drive/MyDrive/enzyme_hybrid_lnn/cache/full_xtb',
        '/content/drive/MyDrive/enzyme_hybrid_lnn/cache/manual_engine_full',
        '/content/drive/MyDrive/enzyme_hybrid_lnn/revisit/checkpoints',
        '/content/drive/MyDrive/enzyme_hybrid_lnn/revisit/artifacts',
    ]:
        os.makedirs(d, exist_ok=True)
    print('Drive mounted.')
except Exception as e:
    print('Drive mount skipped:', e)

In [ ]:
# ── Cell 3: configure revisit parameters ─────────────────────────────────────
import os

# ─── Revisit loop control ─────────────────────────────────────────────────────
os.environ["HYBRID_COLAB_REVISIT_RECENT_REPORTS"] = "5"
os.environ["HYBRID_COLAB_REVISIT_ITERATIONS"]     = "12"
os.environ["HYBRID_COLAB_REVISIT_SELECTION_SCOPE"] = "val"

# ─── Where to write revisit outputs ──────────────────────────────────────────
os.environ["HYBRID_COLAB_REVISIT_OUTPUT_ROOT"]   = "/content/drive/MyDrive/enzyme_hybrid_lnn/revisit/checkpoints"
os.environ["HYBRID_COLAB_REVISIT_ARTIFACT_ROOT"] = "/content/drive/MyDrive/enzyme_hybrid_lnn/artifacts"

# ─── Existing training outputs ────────────────────────────────────────────────
os.environ["HYBRID_COLAB_OUTPUT_DIR"]   = "/content/drive/MyDrive/enzyme_hybrid_lnn/checkpoints/hybrid_full_xtb"
os.environ["HYBRID_COLAB_ARTIFACT_DIR"] = "/content/drive/MyDrive/enzyme_hybrid_lnn/artifacts/hybrid_full_xtb"

# ─── Data paths ───────────────────────────────────────────────────────────────
os.environ["HYBRID_COLAB_DATASET"]       = "data/prepared_training/main7_site_conservative_singlecyp_clean_symm.json"
os.environ["HYBRID_COLAB_STRUCTURE_SDF"] = "3D structures.sdf"
os.environ["HYBRID_COLAB_XTB_CACHE_DIR"]    = "/content/drive/MyDrive/enzyme_hybrid_lnn/cache/full_xtb"
os.environ["HYBRID_COLAB_MANUAL_CACHE_DIR"] = "/content/drive/MyDrive/enzyme_hybrid_lnn/cache/manual_engine_full"

# ─── Training defaults (strategies may override per-attempt) ─────────────────
os.environ["HYBRID_COLAB_LR"]           = "1e-5"   # very low: fine-tunes gently without destroying warm-start
os.environ["HYBRID_COLAB_WD"]           = "1e-4"
os.environ["HYBRID_COLAB_BATCH_SIZE"]   = "16"
os.environ["HYBRID_COLAB_EPOCHS"]       = "50"
os.environ["HYBRID_COLAB_EARLY_STOPPING_PATIENCE"] = "0"   # disabled: run all epochs
os.environ["HYBRID_COLAB_EARLY_STOPPING_METRIC"]   = "site_top1"
os.environ["HYBRID_COLAB_SPLIT_MODE"]  = "scaffold_source_size"
os.environ["HYBRID_COLAB_SEED"]              = "42"
os.environ["HYBRID_COLAB_SITE_LABELED_ONLY"] = "1"

# ─── Analogical memory: UNFROZEN so the memory bank learns useful embeddings ─
# With FREEZE=0, after each epoch the training molecules are re-encoded into
# the memory bank. kNN lookup then returns chemically meaningful neighbors
# instead of random initialized vectors. This is how the analogical voter
# gets smarter over successive revisit iterations.
os.environ["HYBRID_COLAB_FREEZE_NEXUS_MEMORY"] = "0"

# ─── Backbone freeze warmup ───────────────────────────────────────────────────
# Freeze base GNN for first 2 epochs; only train hybrid heads (arbiter/council).
# Short freeze prevents the warm-start from being destroyed by the unfrozen
# memory gradients before heads have adapted.
os.environ["HYBRID_COLAB_BACKBONE_FREEZE_EPOCHS"] = "2"

# ─── Colab torch stability ────────────────────────────────────────────────────
os.environ["TORCHDYNAMO_DISABLE"]         = "1"
os.environ["TORCH_COMPILE_DISABLE"]       = "1"
os.environ["HYBRID_FORCE_MANUAL_OPTIMIZER"] = "1"

# Print active config summary
print("── Revisit configuration ───────────────────────────────────")
for k in [
    "HYBRID_COLAB_REVISIT_RECENT_REPORTS", "HYBRID_COLAB_REVISIT_ITERATIONS",
    "HYBRID_COLAB_REVISIT_SELECTION_SCOPE", "HYBRID_COLAB_OUTPUT_DIR",
    "HYBRID_COLAB_ARTIFACT_DIR", "HYBRID_COLAB_DATASET", "HYBRID_COLAB_LR",
    "HYBRID_COLAB_EPOCHS", "HYBRID_COLAB_EARLY_STOPPING_PATIENCE",
    "HYBRID_COLAB_EARLY_STOPPING_METRIC", "HYBRID_COLAB_SPLIT_MODE",
    "HYBRID_COLAB_FREEZE_NEXUS_MEMORY", "HYBRID_COLAB_BACKBONE_FREEZE_EPOCHS",
]:
    print(f"  {k} = {os.environ.get(k, '(not set)')}")
print("─────────────────────────────────────────────────────────────")


In [ ]:
# ── Cell 4: run the revisit loop ─────────────────────────────────────────────
#
# The script:
#   1. Finds the last 5 episode logs and runs deep diagnostics on each.
#   2. Aggregates findings: overfitting gap, wave degradation, analogical
#      dead-confidence, competition-weight frozen, CYP-dominance, board frozen.
#   3. Builds a priority-ordered strategy list from the evidence.
#   4. For each strategy: train (40-60 epochs) → analyse → compare vs baseline.
#   5. Promotes baseline if improved.
#   6. After 3 consecutive no-improvements: switches to alternate paths
#      (live wave/analogical vote inputs, high-entropy board, memory unlock).
#
import sys
# Force sys.argv to just the script name so argparse doesn't see Jupyter flags
sys.argv = ['/content/enzyme_Software/scripts/colab_revisit_hybrid_lnn.py']
exec(open('/content/enzyme_Software/scripts/colab_revisit_hybrid_lnn.py').read())

In [ ]:
# ── Cell 5: display results summary ──────────────────────────────────────────
import json, os
from pathlib import Path

artifact_root = Path(os.environ.get(
    'HYBRID_COLAB_REVISIT_ARTIFACT_ROOT',
    '/content/drive/MyDrive/enzyme_hybrid_lnn/revisit/artifacts'
))

best_path      = artifact_root / 'revisit_best.json'
bootstrap_path = artifact_root / 'revisit_bootstrap.json'

if best_path.exists():
    best = json.loads(best_path.read_text())
    print('═' * 60)
    print('REVISIT LOOP RESULTS')
    print('═' * 60)
    print(f"  Selection scope:    {best.get('selection_scope', '?')}")
    print(f"  Total attempts:     {best.get('attempt_count', '?')}")
    print(f"  Improvements:       {best.get('improvements', '?')}")
    print(f"  Used alt strategies:{best.get('used_alt_strategies', '?')}")
    print(f"  Best score:         {best.get('baseline_score', 0.0):.6f}")
    print(f"  Best report:        {best.get('baseline_report', '?')}")
    print(f"  Best checkpoint:    {best.get('baseline_checkpoint', '?')}")

if bootstrap_path.exists():
    bs = json.loads(bootstrap_path.read_text())
    attempts = bs.get('attempts', [])
    if attempts:
        print()
        print('── Per-attempt breakdown ──────────────────────────────────')
        print(f"{'#':>3}  {'name':<34} {'score':>8}  {'improved'}")
        print('-' * 60)
        for a in attempts:
            mark = '✓' if a.get('improved') else '✗'
            print(f"{a['attempt']:>3}  {a['name']:<34} {a.get('score', 0.0):>8.4f}  {mark}")
            diag = a.get('diagnostics') or {}
            va = diag.get('voter_accs') or {}
            print(f"     val_top1={diag.get('best_val_top1', 0.0):.3f} "
                  f"val_top3={diag.get('best_val_top3', 0.0):.3f} "
                  f"overfit={diag.get('overfitting_gap', 0.0):.3f} "
                  f"ana_conf={diag.get('ana_conf_mean', 0.0):.3f}")
            if va:
                print(f"     voter_acc lnn={va.get('lnn', 0.0):.3f} "
                      f"wave={va.get('wave', 0.0):.3f} "
                      f"ana={va.get('ana', 0.0):.3f} "
                      f"council={va.get('council', 0.0):.3f}")
else:
    print('No results found yet — has Cell 4 completed?')